# 07b Raw ADC – Execute

Runs the experiment on the QOP and saves `ds_raw` to a local `.nc` file. Use notebook 02 to load and plot.

## Imports

In [15]:
import os
import numpy as np
import xarray as xr
from qm.qua import *

from qualang_tools.multi_user import qm_session
from qualang_tools.units import unit

from iqcc_calibration_tools.qualibrate_config.qualibrate.node import QualibrationNode
from quam_builder.architecture.superconducting.qpu import FluxTunableQuam as Quam
from calibration_utils.iq_blobs_raw_adc import Parameters
from qualibration_libs.parameters import get_qubits
from qualibration_libs.data import XarrayDataFetcher

## Node initialisation

In [16]:
description = """
IQ BLOBS (RAW ADC)
Uses raw ADC traces instead of on-board measure and demodulation.
Measures the resonator N times in |g> state and |e> state.
"""

node = QualibrationNode[Parameters, Quam](
    name="07b_iq_blobs_raw_adc",
    description=description,
    parameters=Parameters(),
)

@node.run_action(skip_if=node.modes.external)
def custom_param(node: QualibrationNode[Parameters, Quam]):
    # node.parameters.qubits = ["qC2"]  # Uncomment to run on subset; leave commented for all qubits
    pass

node.machine = Quam.load()

# Local path for saving ds_raw (same folder as this notebook)
DS_RAW_PATH = os.path.join(os.getcwd(), "07b_iq_blobs_raw_adc_ds_raw_qB5.nc")

# Stream raw ADC only for these qubits (None = stream all). Reduces OPX→host transfer when running on all qubits.
STREAM_RAW_ADC_QUBITS = ["qB5"]

2026-03-04 17:33:42,407 - qualibrate - INFO - Creating node 07b_iq_blobs_raw_adc


Running action custom_param
Action custom_param finished


## Create QUA program

In [17]:
@node.run_action(skip_if=node.parameters.load_data_id is not None)
def create_qua_program(node: QualibrationNode[Parameters, Quam]):
    u = unit(coerce_to_integer=True)
    node.namespace["qubits"] = qubits = get_qubits(node)
    num_qubits = len(qubits)
    qubit_names = qubits.get_names()

    # Indices of qubits for which we stream raw ADC (others use qua_vars only)
    stream_indices = (
        [i for i, n in enumerate(qubit_names) if n in STREAM_RAW_ADC_QUBITS]
        if STREAM_RAW_ADC_QUBITS is not None
        else list(range(num_qubits))
    )
    stream_qubit_names = [qubit_names[i] for i in stream_indices]
    num_stream = len(stream_indices)

    n_runs = node.parameters.num_shots
    operation = node.parameters.operation
    readout_length = qubits[0].resonator.operations[operation].length
    node.namespace["sweep_axes"] = {
        "qubit": xr.DataArray(stream_qubit_names),
        "n_runs": xr.DataArray(np.linspace(1, n_runs, n_runs), attrs={"long_name": "number of shots"}),
        "readout_time": xr.DataArray(
            np.arange(0, readout_length, 1),
            attrs={"long_name": "readout time", "units": "ns"},
        ),
    }

    with program() as node.namespace["qua_program"]:
        n = declare(int)
        n_st = declare_stream()
        # ADC streams only for qubits we stream raw ADC
        adc_g_st = [declare_stream(adc_trace=True) for _ in stream_indices]
        adc_e_st = [declare_stream(adc_trace=True) for _ in stream_indices]

        for multiplexed_qubits in qubits.batch():
            for qubit in multiplexed_qubits.values():
                node.machine.initialize_qpu(target=qubit)
            align()

            with for_(n, 0, n < n_runs, n + 1):
                save(n, n_st)
                for i, qubit in multiplexed_qubits.items():
                    qubit.reset(node.parameters.reset_type, node.parameters.simulate)
                align()
                for i, qubit in multiplexed_qubits.items():
                    reset_if_phase(qubit.resonator.name)
                    if i in stream_indices:
                        qubit.resonator.measure(operation, stream=adc_g_st[stream_indices.index(i)])
                    else:
                        qubit.resonator.play(operation)
                    qubit.resonator.wait(qubit.resonator.depletion_time * u.ns)
                align()

                for i, qubit in multiplexed_qubits.items():
                    qubit.reset(node.parameters.reset_type, node.parameters.simulate)
                align()
                for i, qubit in multiplexed_qubits.items():
                    qubit.xy.play("x180")
                    qubit.align()
                    reset_if_phase(qubit.resonator.name)
                    if i in stream_indices:
                        qubit.resonator.measure(operation, stream=adc_e_st[stream_indices.index(i)])
                    else:
                        qubit.resonator.play(operation)
                    qubit.resonator.wait(qubit.resonator.depletion_time * u.ns)
                align()

        with stream_processing():
            n_st.save("n")
            # Raw ADC stream processing (only for stream qubits)
            for k in range(num_stream):
                stream_g = adc_g_st[k].input1()
                stream_e = adc_e_st[k].input1()
                stream_g.real().buffer(n_runs).save(f"Igraw{k + 1}")
                stream_g.image().buffer(n_runs).save(f"Qgraw{k + 1}")
                stream_e.real().buffer(n_runs).save(f"Ieraw{k + 1}")
                stream_e.image().buffer(n_runs).save(f"Qeraw{k + 1}")

create_qua_program()

Running action create_qua_program
Action create_qua_program finished
Running action create_qua_program
Action create_qua_program finished


In [18]:
node.serialize_qua_program()

c:\Users\gilads\VisualStudioProjects\iqcc_cloud\iqcc-calibration\.venv\Lib\site-packages\quam\components\channels.py:719: UserWarning: The 'thread' element argument is deprecated from qm.qua >= 1.2.2. Use 'core' instead.
  warnings.warn(


2026-03-04 17:34:30,107 - qm - WARNING  - Could not generate a loaded config. Maybe there is no `QuantumMachinesManager` instance?


## Execute

Runs the experiment on the QOP and stores `ds_raw` in the node.

In [14]:
if not node.parameters.simulate and node.parameters.load_data_id is None:
    qmm = node.machine.connect()
    config = node.machine.generate_config()
    with qm_session(qmm, config, timeout=node.parameters.timeout) as qm:
        node.namespace["job"] = job = qm.execute(node.namespace["qua_program"])
        data_fetcher = XarrayDataFetcher(job, node.namespace["sweep_axes"])
        for dataset in data_fetcher:
            pass
        node.log(job.execution_report())

    node.results["ds_raw"] = dataset
    print("Execution finished. ds_raw stored in node.results. Run the next cell to save to file.")
else:
    print("Skipped execution (simulate=True or load_data_id set)")

c:\Users\gilads\VisualStudioProjects\iqcc_cloud\iqcc-calibration\.venv\Lib\site-packages\quam\components\channels.py:719: UserWarning: The 'thread' element argument is deprecated from qm.qua >= 1.2.2. Use 'core' instead.
  warnings.warn(


2026-03-04 16:59:17,441 - qm - INFO     - Opening QM
2026-03-04 16:59:17,451 - qm - WARNING  - Could not generate a loaded config. Maybe there is no `QuantumMachinesManager` instance?
[16:59:27] QUA program submitted to arbel (id = fd95d3c4-02c1-435a-9bf0-3bb324d2b6c5)              ]8;id=972184;file://c:\Users\gilads\VisualStudioProjects\iqcc_cloud\iqcc-calibration\.venv\Lib\site-packages\iqcc_cloud_client\computers.py\computers.py]8;;\:]8;id=488968;file://c:\Users\gilads\VisualStudioProjects\iqcc_cloud\iqcc-calibration\.venv\Lib\site-packages\iqcc_cloud_client\computers.py#314\314]8;;\
           Execution started                                                                       ]8;id=154810;file://c:\Users\gilads\VisualStudioProjects\iqcc_cloud\iqcc-calibration\.venv\Lib\site-packages\iqcc_cloud_client\computers.py\computers.py]8;;\:]8;id=604796;file://c:\Users\gilads\VisualStudioProjects\iqcc_cloud\iqcc-calibration\.venv\Lib\site-packages\iqcc_cloud_client\computers

AttributeError: 'NoneType' object has no attribute 'is_processing'

## Save to file

Saves `ds_raw` to a local `.nc` file for loading in notebook 02.

In [ ]:
if "ds_raw" in node.results:
    node.results["ds_raw"].to_netcdf(DS_RAW_PATH)
    print(f"Saved ds_raw to {DS_RAW_PATH}")
else:
    print("No ds_raw in node.results. Run the Execute cell first.")

Saved ds_raw to c:\Users\gilads\VisualStudioProjects\iqcc_cloud\07b_iq_blobs_raw_adc_ds_raw_qB5nc
